In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")
%matplotlib inline

# Загружаем данные
df = pd.read_csv('../data/dataset.csv')
df.head()

,Customer_ID,Churn_Label,Age,Gender,Location_Region,Marital_Status,Annual_Income,Membership_Tier,Join_Date,Last_Login_Date,Total_Spend_Lifetime,Items_In_Cart,Avg_Session_Duration_Min,Customer_Support_Calls,Preferred_Device,Payment_Method,Marketing_Opt_In,Last_Feedback_Text,Discount_Code_Used
0,CUST_00000,0,45,Female,West,Divorced,79239.91,Silver,2021-10-16,2023-11-02,944.99,6,12.1,0,Desktop,COD,True,Great service!,NaN
1,CUST_00001,1,24,Female,East,Married,67390.43,Platinum,2020-04-24,2023-01-02,45.50,5,19.1,2,Tablet,PayPal,False,Too expensive.,SAVE10
2,CUST_00002,0,52,Female,East,Married,96437.88,Platinum,2020-01-26,2023-05-22,1394.82,13,9.1,1,Mobile,PayPal,False,Worst experience ever.,SUMMER20
3,CUST_00003,0,45,Male,South,Married,31695.81,Platinum,2022-01-29,2023-03-16,82.44,0,11.9,2,Tablet,PayPal,True,Great service!,SAVE10
4,CUST_00004,0,31,Female,East,Divorced,47409.48,Gold,2020-10-08,2023-03-08,1068.49,16,6.1,2,Tablet,Credit Card,False,Too expensive.,NaN


# Лабораторная работа №1.1 (Продолжение). 

## Задание 1. Типы признаков и шкалы измерения

**1. Типы шкал для каждого признака:**
*   `Customer_ID` — **Номинальная** (уникальный номер, математические операции бессмысленны).
*   `Churn_Label` — **Номинальная** (бинарный признак: ушел/остался).
*   `Age`, `Annual_Income`, `Total_Spend_Lifetime` — **Шкала отношений** (есть абсолютный ноль: 0 лет, 0 рублей; можно сказать, что доход в 2 раза больше).
*   `Membership_Tier` (Silver/Gold/Platinum) — **Порядковая** (есть порядок уровней, но нет равных интервалов).
*   `Join_Date`, `Last_Login_Date` — **Интервальная** (можно вычислить разницу — стаж в днях).

**2. Два признака с разными шкалами: `Age` (отношений) и `Gender` (номинальная).**
*   **Среднее:** Для `Age` — **имеет смысл** (средний возраст). Для `Gender` — **бессмысленно** (нельзя сложить "Мужчина" и "Женщина" и поделить на 2).
*   **Медиана:** Для `Age` — **имеет смысл** (возраст, делящий выборку пополам). Для `Gender` — **не имеет** (медиана просто покажет самую частую категорию, но это не дает информации о центре).
*   **Мода:** Для `Gender` — **идеально** (покажет, кого больше). Для `Age` — **имеет смысл, но малополезно** (возраст может быть 35 лет, но таких людей может быть всего 2% от выборки).
*   **Корректные агрегаты:** Для `Age` — сумма, среднее, дисперсия. Для `Gender` — только подсчет частот (`value_counts()`).
*   **Некорректные:** Если применить `.mean()` к полу (закодированному цифрами 1 и 2), pandas выдаст `1.5`, что не имеет физического смысла.

## Задание 2. Пропуски и их природа
Выбираем признак `Discount_Code_Used` (где 25% пропусков).

1. **Гипотеза MCAR (Пропуск абсолютно случайный):** 
Пропуски возникли из-за сбоя выгрузки. Факт пропуска не связан ни с какими другими признаками.
* *Как проверить:* Сравнить распределение возраста и трат у клиентов с пропусками и без. Если они идентичны — гипотеза верна.

2. **Гипотеза MAR (Пропуск случайный, но зависит от других признаков):** 
Пропуски зависят от другого признака. Например, клиенты с низким доходом не получают промокоды, поэтому в базе пусто.
* *Как проверить:* Построить график зависимости доли пропусков от уровня дохода (`Annual_Income`).

3. **Гипотеза MNAR (Пропуск не случайный, зависит от скрытого значения):** 
Самые богатые (VIP) клиенты не пользуются промокодами, и система не записывает это.
* *Как проверить:* Проверить пересечение с `Membership_Tier`. Если у Platinum пропусков больше — возможно, это MNAR.

## Задание 3. Индикатор пропуска
1. **Когда факт пропуска несёт информацию:** Когда пропуск не случаен. Например, если клиент не указал `Last_Feedback_Text`, это сигнал: он пассивный и не вовлечён.
2. **Почему значимость индикатора — это сигнал о плохой импутации:** Если мы заполнили пропуски медианой, появилась искусственная куча одинаковых значений. Модель начнет "цепляться" за факт пропуска, хотя мы подставили заглушку.
3. **Примеры:**
* **Полезен:** `Discount_Code_Used` (отсутствие промокода может влиять на лояльность).
* **Бесполезен:** `Gender` (пропуск пола вряд ли связан с покупками).
* **Опасен:** `Total_Spend_Lifetime` (если заполнить нулем, модель подумает, что это новичок с нулевыми тратами).

## Задание 4. Выброс — это ошибка или сигнал?
Выбираем признак `Annual_Income`. Найден выброс — значение около **120 000**.

1. **Реальные причины:** 
* Это **сигнал**: VIP-клиент или бизнесмен с высоким доходом. Это реальный сегмент.
* Это **ошибка**: Кто-то случайно нажал лишний ноль при регистрации.
2. **Что хуже для анализа:**
* **Удалить:** Плохо, потеряем важнейший сегмент "богатых" клиентов.
* **Оставить без изменений:** Плохо для линейных моделей (одно значение сдвинет прямую).
* **Применить преобразование (логарифм):** **Лучший вариант**. Сожмёт масштаб, сохранив разницу между обычным и VIP-клиентом.

## Задание 5. Преобразование или винизоризация?
Для `Total_Spend_Lifetime` (сильная асимметрия).

1. **Логарифмирование:** Когда данные имеют экспоненциальный рост (траты). Помогает приблизить к нормальному.
2. **Корень:** Когда разброс не гигантский, мягче сжимает.
3. **Винизоризация:** Когда выбросы — явные ошибки, обрезаем до 95-го перцентиля.
* **Влияние на интерпретируемость:** После логарифма мы не можем сказать "потратил 1000 рублей", мы говорим "log(траты) = 3".
* **Влияние на модели:** Логарифм спасает линейные модели от выбросов.

## Задание 7. Неправильная диаграмма
Из работы возьмем график распределения возрастов (`Age`).
1. **Хуже выбранный тип:** **Круговая диаграмма (Pie chart)** для 20+ возрастных категорий.
2. **Что исказит:** Углы и площади секторов очень трудно сравнивать на глаз.
3. **Какой неверный вывод:** Зритель может решить, что клиенты старше 60 лет — большая группа, хотя на самом деле их 2%, просто сектор вытянулся.

## Задание 8. Одна и та же информация — разные графики
Для пары `Age` и `Annual_Income`:
1. **Точечный график (Scatterplot):** показывает каждую точку. Подчеркивает разброс, кластеры, выбросы.
2. **Boxplot:** покажет медиану дохода в разных возрастных группах. 
* **Информация:** Скаттер сразу показывает VIP-клиентов (единицы). Боксплот скроет выбросы, но четко покажет "ядро" — типичный доход.

In [3]:
# Задание 6
col = 'Annual_Income'
arith_mean = df[col].mean()
median = df[col].median()
geo_mean = np.exp(np.log(df[col][df[col] > 0]).mean()) # защита от логарифма нуля

print(f"Арифметическое среднее: {arith_mean:.2f}")
print(f"Медиана: {median:.2f}")
print(f"Геометрическое среднее: {geo_mean:.2f}")

Арифметическое среднее: 54969.30
Медиана: 55310.21
Геометрическое среднее: 52682.27


**Объяснение к Заданию 6:** 
Они различаются, потому что в данных есть выбросы (очень богатые люди). Арифметическое среднее "тянется" вверх из-за них. Медиана лучше отражает "типичного" клиента, так как она устойчива к выбросам. Геометрическое среднее здесь ближе к медиане.